In [ ]:
import os
from dataclasses import dataclass
DEFAULT_TUSHARE_TOKEN = os.environ.get('TUSHARE_TOKEN', '')
DEFAULT_TUSHARE_BASE_URL = "https://tt.xiaodefa.cn"
DEFAULT_TUSHARE_TIMEOUT = 10
DEFAULT_TUSHARE_MIN_INTERVAL_SECONDS = 0.55

@dataclass(frozen=True)
class Research:
    start_date:str
    end_date:str
    benchmark:str = "000001.SH"
    base_url:str = DEFAULT_TUSHARE_BASE_URL
    timeout:int = DEFAULT_TUSHARE_TIMEOUT
    min_seconds:float = DEFAULT_TUSHARE_MIN_INTERVAL_SECONDS

In [ ]:
from datetime import datetime
def valid_date_range(
        config: Research
) -> None:
    start_date = config.start_date
    end_date = config.end_date
    if start_date > end_date:
        raise ValueError("开始日期不能晚于结束日期")

In [ ]:
import tushare as ts
def create_tushare_client(
        config: Research
):
    client = ts.pro_api(
        DEFAULT_TUSHARE_TOKEN,
        timeout=config.timeout,
    )
    client._DataApi__http_url = config.base_url.rstrip("/")
    return client


In [ ]:
config = Research(
    start_date="20251124",
    end_date="20260722",
    benchmark="000001.SH",
)
valid_date_range(config)
client = create_tushare_client(config)
trade_calendar = client.query(
    "trade_cal",
    start_date="20251124",
    end_date="20260722",
    is_open = "1",
)
trade_calendar.head()

In [ ]:
from datetime import timedelta
import pandas as pd
def fetch_index(
        client,
        config: Research,
        warmup_days: int = 240,
) -> pd.DataFrame:
    start_date = datetime.strptime(config.end_date, "%Y%m%d")
    query_start = start_date - timedelta(days=warmup_days)
    query_start_desc = datetime.strftime(query_start, "%Y%m%d")
    frame = client.query(
        "idx_factor_pro",
        ts_code=config.benchmark,
        start_date=query_start_desc,
        end_date=config.end_date,
    )

    if frame.empty:
        raise RuntimeError(f"未获取到数据，日期：{query_start_desc}，标的：{config.benchmark}")
    return frame.sort_values("trade_date", ascending=False).reset_index(drop=True)

In [ ]:
frame = (
    fetch_index(client, config)
    .sort_values("trade_date")
    .reset_index(drop=True)
)
frame["return_5d"] = frame["close"].pct_change(5)
frame["ma20_slope_5d"] = frame["ma_bfq_20"].pct_change(5)
frame.head()

In [ ]:
import time
TARGET_KEY_MAN = "国泰海通证券股份有限公司武汉紫阳东路证券营业部"
def fetch_branch_buy_rows(
        client,
        trade_calendar:pd.DataFrame,
        config: Research,
) -> pd.DataFrame:
    buy_frame = []
    open_dates = trade_calendar.loc[
    trade_calendar["is_open"].astype(int) == 1,
    "cal_date",
    ].astype(str)
    for index,trade_date in enumerate(open_dates):
        if index > 0:
            time.sleep(config.min_seconds)
        daily_frame = client.query(
            "top_inst",
            trade_date = trade_date,
        )
        if daily_frame.empty:
            continue
        branch_mask = daily_frame["exalter"].fillna("").str.contains(
            TARGET_KEY_MAN,
            regex=False,
        )
        buy_mask = daily_frame["side"].astype(str).eq("0")
        matched = daily_frame.loc[branch_mask&buy_mask].copy()
        if not matched.empty:
            buy_frame.append(matched)
    if not buy_frame:
        return pd.DataFrame()
    return (
        pd.concat(buy_frame, ignore_index=True)
        .sort_values(["trade_date", "ts_code"])
        .reset_index(drop=True)
    )

In [ ]:
branch_buy_raws = fetch_branch_buy_rows(
    client,
        trade_calendar,
    config,
)
buy_dates = (
    branch_buy_raws["trade_date"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)
buy_dates

In [ ]:
import numpy as np
WINDOW_PARTITION = (
    "连续三个交易日|连续3个交易日"
    "三个交易日内|3个交易日内|严重异常波动"
)
buy_rows = branch_buy_raws.copy()
buy_rows["buy"] = pd.to_numeric(buy_rows["buy"], errors="coerce")
buy_rows["event_type"] = np.where(
    buy_rows["reason"].fillna("").str.contains(WINDOW_PARTITION,regex=True),
    "WINDOW_BUY",
    "EXACT_BUY",
)
stock_events = (
    buy_rows
    .groupby(["ts_code","trade_date","event_type"],as_index=False)
    .agg(buy_amount = ("buy","max"))
)
daily_events = (
    stock_events
    .groupby(["trade_date","event_type"],as_index=False)
    .agg(buy_amount = ("buy_amount","sum"),
         stock_count = ("ts_code","unique")
         )
)

daily_events

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

WINDOW = 5
index_plot = (
    frame.loc[frame["trade_date"].between(config.start_date, config.end_date)]
    .sort_values("trade_date")
    .reset_index(drop=True)
)
date_to_pos = pd.Series(index_plot.index, index=index_plot["trade_date"]).to_dict()

# 同一天的不同榜单类型先合并；只保留买入金额前25%的高强度事件。
event_days = (
    daily_events.groupby("trade_date", as_index=False)
    .agg(buy_amount=("buy_amount", "sum"))
)
amount_cutoff = event_days["buy_amount"].quantile(0.75)
candidates = event_days.loc[event_days["buy_amount"] >= amount_cutoff].copy()
candidates["position"] = candidates["trade_date"].map(date_to_pos)
candidates = candidates.dropna(subset=["position"])
candidates["position"] = candidates["position"].astype(int)

# 按金额从大到小挑选，并排除相隔3个交易日以内的事件，避免连续披露重复计权。
selected_rows = []
selected_positions = []
for row in candidates.sort_values("buy_amount", ascending=False).itertuples():
    if all(abs(row.position - pos) > 3 for pos in selected_positions):
        if WINDOW <= row.position < len(index_plot) - WINDOW:
            selected_rows.append(row)
            selected_positions.append(row.position)
selected_events = pd.DataFrame(selected_rows).sort_values("trade_date").reset_index(drop=True)

relative_days = np.arange(-WINDOW, WINDOW + 1)
def normalized_window(position):
    prices = index_plot.loc[position - WINDOW:position + WINDOW, "close"].to_numpy()
    return (prices / prices[WINDOW] - 1) * 100

event_matrix = np.vstack([normalized_window(pos) for pos in selected_events["position"]])
all_positions = range(WINDOW, len(index_plot) - WINDOW)
baseline_matrix = np.vstack([normalized_window(pos) for pos in all_positions])
prior_returns = index_plot.loc[selected_events["position"], "return_5d"].to_numpy()
down_mask = prior_returns < 0

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)
ax = axes[0]
for path in event_matrix:
    ax.plot(relative_days, path, color="#D85845", alpha=0.16, linewidth=1)
ax.plot(relative_days, np.median(event_matrix, axis=0), color="#C93F2C", linewidth=3, label="高强度事件中位数")
ax.plot(relative_days, np.median(baseline_matrix, axis=0), color="#355C7D", linewidth=2, linestyle="--", label="全部交易日基准")
ax.set_title(f"去重后的高强度买入事件（n={len(event_matrix)}）")
ax.set_ylabel("相对T日收盘收益（%）")
ax.legend(frameon=False)

ax = axes[1]
if down_mask.any():
    ax.plot(relative_days, np.median(event_matrix[down_mask], axis=0), color="#2F7D71", linewidth=3, label=f"买入前5日下跌（n={down_mask.sum()}）")
if (~down_mask).any():
    ax.plot(relative_days, np.median(event_matrix[~down_mask], axis=0), color="#D08B32", linewidth=3, label=f"买入前5日上涨（n={(~down_mask).sum()}）")
ax.plot(relative_days, np.median(baseline_matrix, axis=0), color="#355C7D", linewidth=1.8, linestyle="--", label="全部交易日基准")
ax.set_title("按买入前5日指数状态分组")
ax.legend(frameon=False)

for ax in axes:
    ax.axvline(0, color="#222222", linewidth=1, linestyle=":")
    ax.axhline(0, color="#999999", linewidth=0.8)
    ax.set_xlabel("相对事件日（T）")
    ax.set_xticks(relative_days)
    ax.grid(alpha=0.18)

fig.suptitle("紫阳东路高强度买入：上证指数事件窗口", fontsize=16, y=1.02)
fig.tight_layout()
plt.show()

selected_events[["trade_date", "buy_amount"]]